In [1]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse, mae
import numpy as np
import math
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [3]:


# Đọc file ratings.dat
ratings_df = pd.read_csv('ratings.dat', sep='::', engine='python', 
                      names=['UserId', 'MovieId', 'Rating', 'Timestamp'])

# Đọc file tags.dat
tags_df = pd.read_csv('tags.dat', sep='::', engine='python', 
                   names=['UserId', 'MovieId', 'Tag', 'Timestamp'])

# Đọc file movies.dat
movies_df = pd.read_csv('movies.dat', sep='::', engine='python', 
                     names=['MovieId', 'Title', 'Genres'])

In [4]:
# Loại bỏ các hàng có dữ liệu thiếu
ratings_df.dropna(inplace=True)
tags_df.dropna(inplace=True)
movies_df.dropna(inplace=True)

In [5]:
from sklearn.model_selection import train_test_split

# Load
df = ratings_df[["UserId", "MovieId", "Rating"]].copy()

# 80% train, 20% temp
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)

# 10% val, 10% test
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(len(train_df), len(val_df), len(test_df))

8000043 1000005 1000006


In [ ]:
from surprise import Dataset, Reader, SVD

reader = Reader(rating_scale=(0.5, 5.0))

train_data = Dataset.load_from_df(
    train_df[["UserId", "MovieId", "Rating"]],
    reader
)

trainset = train_data.build_full_trainset()

model2 = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

model2.fit(trainset)

In [12]:
from surprise import SVDpp

model = SVDpp(
    n_factors=40,
    n_epochs=12,
    lr_all=0.007,
    reg_all=0.02,
    random_state=42
)
model.fit(trainset)

In [ ]:
from sklearn.metrics import mean_squared_error

def evaluate_rmse(df_eval):
    y_true = []
    y_pred = []

    for _, row in df_eval.iterrows():
        pred = model.predict(row["UserId"], row["MovieId"]).est
        y_true.append(row["Rating"])
        y_pred.append(pred)

    return np.sqrt(mean_squared_error(y_true, y_pred))


print("Val RMSE:", evaluate_rmse(val_df))


Val RMSE: 0.7984773460940496
Test RMSE: 0.7977117207648723


In [8]:
from sklearn.metrics import ndcg_score

def evaluate_ndcg(df_eval, k=10):
    user_groups = df_eval.groupby("UserId")
    ndcg_list = []

    for user, group in user_groups:
        true_scores = group["Rating"].values.reshape(1, -1)
        pred_scores = np.array([
            model.predict(user, item).est for item in group["MovieId"]
        ]).reshape(1, -1)

        if len(true_scores[0]) > 1:
            ndcg = ndcg_score(true_scores, pred_scores, k=k)
            ndcg_list.append(ndcg)

    return np.mean(ndcg_list)


print("Val NDCG@10:", evaluate_ndcg(val_df))
print("Test NDCG@10:", evaluate_ndcg(test_df))

Val NDCG@10: 0.9509386742085427
Test NDCG@10: 0.9508333590963857


In [ ]:
def evaluate_ranking_metrics(df_eval, k=10, threshold=4.0):
    user_groups = df_eval.groupby("UserId")
    
    precisions = []
    recalls = []
    hit_rates = []
    
    for user, group in user_groups:
        item_preds = []
        
        # Dự đoán điểm cho tất cả các item của user trong tập eval
        for _, row in group.iterrows():
            est = model.predict(user, row["MovieId"]).est
            item_preds.append((est, row["Rating"]))
            
        # Sắp xếp giảm dần theo điểm dự đoán (est)
        item_preds.sort(key=lambda x: x[0], reverse=True)
        
        # Lấy Top K item được model đánh giá cao nhất
        top_k = item_preds[:k]
        
        # Đếm số lượng item thực sự relevant (thực tế user đánh giá >= threshold)
        n_rel = sum((true_r >= threshold) for (_, true_r) in item_preds)
        
        # Đếm số lượng item relevant xuất hiện trong Top K (Hits)
        n_rel_and_rec_k = sum((true_r >= threshold) for (_, true_r) in top_k)
        
        # Chỉ tính toán cho các user thực sự có item relevant trong tập eval này
        if n_rel > 0:
            # Precision@K: Tỉ lệ dự đoán đúng trong Top K
            precisions.append(n_rel_and_rec_k / len(top_k))
            
            # Recall@K: Tỉ lệ tìm được so với tổng số item relevant của user đó
            recalls.append(n_rel_and_rec_k / n_rel)
            
            # HitRate@K: Bằng 1 nếu có ít nhất 1 item relevant lọt vào Top K, ngược lại là 0
            hit_rates.append(1 if n_rel_and_rec_k > 0 else 0)
            
    return np.mean(precisions), np.mean(recalls), np.mean(hit_rates)

# Chạy thử trên tập Val và Test
val_precision, val_recall, val_hit_rate = evaluate_ranking_metrics(val_df, k=10, threshold=4.0)
test_precision, test_recall, test_hit_rate = evaluate_ranking_metrics(test_df, k=10, threshold=4.0)



Val Precision@10: 0.6645 | Val Recall@10: 0.8552 | Val HitRate@10: 0.9996
Test Precision@10: 0.6632 | Test Recall@10: 0.8551 | Test HitRate@10: 0.9997


In [10]:
print(f"Val Precision@10: {val_precision:.4f} | Val Recall@10: {val_recall:.4f} | Val HitRate@10: {val_hit_rate:.4f}")
print(f"Test Precision@10: {test_precision:.4f} | Test Recall@10: {test_recall:.4f} | Test HitRate@10: {test_hit_rate:.4f}")
print("Val NDCG@10:", evaluate_ndcg(val_df))
print("Test NDCG@10:", evaluate_ndcg(test_df))
print("Val RMSE:", evaluate_rmse(val_df))


Val Precision@10: 0.6645 | Val Recall@10: 0.8552 | Val HitRate@10: 0.9996
Test Precision@10: 0.6632 | Test Recall@10: 0.8551 | Test HitRate@10: 0.9997
Val NDCG@10: 0.9509386742085427
Test NDCG@10: 0.9508333590963857
Val RMSE: 0.7984773460940496
